In [ ]:
# Change this to your preferred framework (e.g., 'cuda', 'pytorch', 'triton', 'jax', 'mojo')
EVAL_LANG = 'cuda'

SAVE_GPU = True


<p>
  Implement a GPU program to calculate the Mean Squared Error (MSE) between
  predicted values and target values. Given two arrays of equal length,
  <code>predictions</code> and <code>targets</code>, compute: $$ \text{MSE} =
  \frac{1}{N} \sum_{i=1}^{N} (predictions_i - targets_i)^2 $$ where N is the
  number of elements in each array.
</p>

<h2>Implementation Requirements</h2>
<ul>
  <li>External libraries are not permitted.</li>
  <li>The <code>solve</code> function signature must remain unchanged.</li>
  <li>The final result must be stored in the <code>mse</code> variable.</li>
</ul>

<h2>Example 1:</h2>
<pre>
  Input:  predictions = [1.0, 2.0, 3.0, 4.0]
          targets = [1.5, 2.5, 3.5, 4.5]
  Output: mse = 0.25
</pre>

<h2>Example 2:</h2>
<pre>
  Input:  predictions = [10.0, 20.0, 30.0]
          targets = [12.0, 18.0, 33.0]
  Output: mse = 5.67
</pre>

<h2>Constraints</h2>
<ul>
  <li>1 &le; <code>N</code> &le; 100,000,000</li>
  <li>
    -1000.0 &le; <code>predictions[i]</code>, <code>targets[i]</code> &le;
    1000.0
  </li>

  <li>Performance is measured with <code>N</code> = 50,000,000</li>
</ul>


# CUDA

In [ ]:
%%writefile solution.cu
#include <cuda_runtime.h>

// predictions, targets, mse are device pointers
extern "C" void solve(const float* predictions, const float* targets, float* mse, int N) {}


# CUTE

In [ ]:
%%writefile solution.py
import cutlass
import cutlass.cute as cute


# predictions, targets, mse are tensors on the GPU
@cute.jit
def solve(predictions: cute.Tensor, targets: cute.Tensor, mse: cute.Tensor, N: cute.Int32):
    pass


# JAX

In [ ]:
%%writefile solution.py
import jax
import jax.numpy as jnp


# predictions, targets are tensors on the GPU
@jax.jit
def solve(predictions: jax.Array, targets: jax.Array, N: int) -> jax.Array:
    # return output tensor directly
    pass


# MOJO

In [ ]:
%%writefile solution.mojo
from std.gpu.host import DeviceContext
from std.gpu import block_dim, block_idx, thread_idx
from std.memory import UnsafePointer
from std.math import ceildiv


@export
def solve(
    predictions: UnsafePointer[Float32, MutExternalOrigin],
    targets: UnsafePointer[Float32, MutExternalOrigin],
    mse: UnsafePointer[Float32, MutExternalOrigin],
    N: Int32,
) raises:
    pass


# Torch

In [ ]:
%%writefile solution.py
import torch


# predictions, targets, mse are tensors on the GPU
def solve(predictions: torch.Tensor, targets: torch.Tensor, mse: torch.Tensor, N: int):
    pass


# Triton

In [ ]:
%%writefile solution.py
import torch
import triton
import triton.language as tl


# predictions, targets, mse are tensors on the GPU
def solve(predictions: torch.Tensor, targets: torch.Tensor, mse: torch.Tensor, N: int):
    pass


# Evaluate Setup

In [ ]:
# --- Core Challenge Base ---
from abc import ABC, abstractmethod
from typing import Any, Dict, List


class ChallengeBase(ABC):
    def __init__(self, name: str, atol: float, rtol: float, num_gpus: int, access_tier: str):
        self.name = name
        self.atol = atol
        self.rtol = rtol
        self.num_gpus = num_gpus
        self.access_tier = access_tier

    @abstractmethod
    def reference_impl(self, *args, **kwargs):
        """
        Reference solution implementation.
        """
        pass

    @abstractmethod
    def get_solve_signature(self) -> Dict[str, Any]:
        """
        Get the function signature for solution.

        Returns:
            Dictionary with argtypes and restype for ctypes
        """
        pass

    @abstractmethod
    def generate_example_test(self) -> List[Dict[str, Any]]:
        """
        Generate an example test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass

    @abstractmethod
    def generate_functional_test(self) -> List[Dict[str, Any]]:
        """
        Generate functional test cases for this problem.

        Returns:
            List of test case dictionaries
        """
        pass

    @abstractmethod
    def generate_performance_test(self) -> List[Dict[str, Any]]:
        """
        Generate a performance test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass


# --- Challenge Logic ---
import ctypes
from typing import Any, Dict, List

import torch


class Challenge(ChallengeBase):
    def __init__(self):
        super().__init__(
            name="Mean Squared Error", atol=1e-05, rtol=1e-05, num_gpus=1, access_tier="free"
        )

    def reference_impl(
        self, predictions: torch.Tensor, targets: torch.Tensor, mse: torch.Tensor, N: int
    ):
        # predictions, targets, mse are tensors on the GPU
        squared_diffs = torch.square(predictions - targets)
        mean_squared_error = torch.mean(squared_diffs)
        mse[0] = mean_squared_error

    def get_solve_signature(self) -> Dict[str, tuple]:
        return {
            "predictions": (ctypes.POINTER(ctypes.c_float), "in"),
            "targets": (ctypes.POINTER(ctypes.c_float), "in"),
            "mse": (ctypes.POINTER(ctypes.c_float), "out"),
            "N": (ctypes.c_int, "in"),
        }

    def generate_example_test(self) -> Dict[str, Any]:
        dtype = torch.float32
        predictions = torch.tensor([1.0, 2.0, 3.0, 4.0], device="cuda", dtype=dtype)
        targets = torch.tensor([1.5, 2.5, 3.5, 4.5], device="cuda", dtype=dtype)
        mse = torch.empty(1, device="cuda", dtype=dtype)
        N = 4
        return {
            "predictions": predictions,
            "targets": targets,
            "mse": mse,
            "N": N,
        }

    def generate_functional_test(self) -> List[Dict[str, Any]]:
        dtype = torch.float32
        tests = []
        # Test 1: basic_example
        tests.append(
            {
                "predictions": torch.tensor([1.0, 2.0, 3.0, 4.0], device="cuda", dtype=dtype),
                "targets": torch.tensor([1.5, 2.5, 3.5, 4.5], device="cuda", dtype=dtype),
                "mse": torch.empty(1, device="cuda", dtype=dtype),
                "N": 4,
            }
        )
        # Test 2: second_example
        tests.append(
            {
                "predictions": torch.tensor([10.0, 20.0, 30.0], device="cuda", dtype=dtype),
                "targets": torch.tensor([12.0, 18.0, 33.0], device="cuda", dtype=dtype),
                "mse": torch.empty(1, device="cuda", dtype=dtype),
                "N": 3,
            }
        )
        # Test 3: zero_error
        tests.append(
            {
                "predictions": torch.tensor([1.5, 2.5, 3.5, 4.5, 5.5], device="cuda", dtype=dtype),
                "targets": torch.tensor([1.5, 2.5, 3.5, 4.5, 5.5], device="cuda", dtype=dtype),
                "mse": torch.empty(1, device="cuda", dtype=dtype),
                "N": 5,
            }
        )
        # Test 4: negative_values
        tests.append(
            {
                "predictions": torch.tensor([-2.5, -1.0, 0.0, 1.5], device="cuda", dtype=dtype),
                "targets": torch.tensor([-1.5, -2.0, 0.5, 2.0], device="cuda", dtype=dtype),
                "mse": torch.empty(1, device="cuda", dtype=dtype),
                "N": 4,
            }
        )
        # Test 5: large_difference
        tests.append(
            {
                "predictions": torch.tensor([100.0, 200.0, 300.0], device="cuda", dtype=dtype),
                "targets": torch.tensor([150.0, 250.0, 350.0], device="cuda", dtype=dtype),
                "mse": torch.empty(1, device="cuda", dtype=dtype),
                "N": 3,
            }
        )
        # Test 6: medium_size
        N = 1024
        predictions = torch.empty(N, device="cuda", dtype=dtype).uniform_(-10.0, 10.0)
        targets = torch.empty(N, device="cuda", dtype=dtype).uniform_(-10.0, 10.0)
        mse = torch.empty(1, device="cuda", dtype=dtype)
        tests.append({"predictions": predictions, "targets": targets, "mse": mse, "N": N})
        # Test 7: large_size
        N = 10000
        predictions = torch.empty(N, device="cuda", dtype=dtype).uniform_(-100.0, 100.0)
        targets = torch.empty(N, device="cuda", dtype=dtype).uniform_(-100.0, 100.0)
        mse = torch.empty(1, device="cuda", dtype=dtype)
        tests.append({"predictions": predictions, "targets": targets, "mse": mse, "N": N})
        return tests

    def generate_performance_test(self) -> Dict[str, Any]:
        dtype = torch.float32
        N = 50000000
        predictions = torch.empty(N, device="cuda", dtype=dtype).uniform_(-1000.0, 1000.0)
        targets = torch.empty(N, device="cuda", dtype=dtype).uniform_(-1000.0, 1000.0)
        mse = torch.empty(1, device="cuda", dtype=dtype)
        return {
            "predictions": predictions,
            "targets": targets,
            "mse": mse,
            "N": N,
        }


ch = Challenge()



In [ ]:
import os
import time
import ctypes
import torch

class Evaluate:
    @staticmethod
    def eval_cuda(ch):
        # 1. Compile a fresh uniquely named library
        so_filename = f'solution_func_{int(time.time())}.so'
        os.system(f'nvcc -shared -Xcompiler -fPIC -O3 solution.cu -o {so_filename}')
        lib = ctypes.CDLL(f'./{so_filename}')
        
        # 2. Extract signature and set argtypes
        signature = ch.get_solve_signature()
        lib.solve.argtypes = [arg_info[0] for arg_info in signature.values()]
        
        Evaluate._run_tests(ch, signature, lambda kwargs: lib.solve(*Evaluate._build_cuda_args(kwargs, signature)))

    @staticmethod
    def eval_python(ch):
        import importlib.util
        import sys
        
        spec = importlib.util.spec_from_file_location("solution", "solution.py")
        solution = importlib.util.module_from_spec(spec)
        sys.modules["solution"] = solution
        spec.loader.exec_module(solution)
        
        signature = ch.get_solve_signature()
        Evaluate._run_tests(ch, signature, lambda kwargs: Evaluate._run_python(solution, kwargs))

    @staticmethod
    def _run_python(solution, kwargs):
        solution.solve(**kwargs)
        if torch.cuda.is_available():
            torch.cuda.synchronize()

    @staticmethod
    def eval_mojo(ch):
        print("Mojo evaluation is currently executed via a separate runner or wrapper.")
        print("Ensure you have the mojo compiler installed and use 'mojo build solution.mojo' + ctypes/ffi,")
        print("or run an external python bridge. This is a stub.")

    @staticmethod
    def _build_cuda_args(kwargs, signature):
        cuda_args = []
        for k, (arg_type, dir_type) in signature.items():
            val = kwargs[k]
            if isinstance(val, torch.Tensor):
                cuda_args.append(ctypes.cast(val.data_ptr(), arg_type))
            else:
                cuda_args.append(arg_type(val))
        return cuda_args

    @staticmethod
    def _run_tests(ch, signature, run_fn):
        print("=== Running Functional Tests ===")
        functional_tests = ch.generate_functional_test()
        all_passed = True
        
        for i, test in enumerate(functional_tests):
            ref_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            test_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            
            # Run Reference
            ch.reference_impl(**ref_kwargs)
            
            # Run implementation
            run_fn(test_kwargs)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            
            # Verify outputs
            match = True
            for k, (_, dir_type) in signature.items():
                if dir_type == "out":
                    if not torch.allclose(ref_kwargs[k], test_kwargs[k], atol=ch.atol, rtol=ch.rtol):
                        match = False
                        print(f"❌ Test {i+1}/{len(functional_tests)} Failed on output '{k}'")
                        break
            
            if match:
                print(f"✅ Test {i+1}/{len(functional_tests)} Passed")
            else:
                all_passed = False
                break
                
        if all_passed:
            print("\n🎉 All functional tests passed!")
            return True
        else:
            return False



# Evaluation code

In [ ]:
# Run the evaluator based on configuration
if EVAL_LANG == 'cuda':
    Evaluate.eval_cuda(ch)
elif EVAL_LANG in ['pytorch', 'triton', 'jax', 'cute']:
    Evaluate.eval_python(ch)
elif EVAL_LANG == 'mojo':
    Evaluate.eval_mojo(ch)
else:
    print(f"Unknown language {EVAL_LANG}")

# Disconnect runtime to save Colab resources
if SAVE_GPU:
    from google.colab import runtime
    runtime.unassign()
